In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import html

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2

from nltk.stem import SnowballStemmer

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
TRAIN_FILE = 'winter_project_2026/development.csv'   
EVAL_FILE = 'winter_project_2026/evaluation.csv'     
OUTPUT_FILE = 'winter_project_2026/sample_submission.csv'

dev_df = pd.read_csv(TRAIN_FILE)
eval_df = pd.read_csv(EVAL_FILE)

dev_df.head()

In [ ]:
dev_df.info()

In [ ]:
# Labels' mapping  
label_map = {
    0: 'International', 1: 'Business', 2: 'Technology', 
    3: 'Entertainment', 4: 'Sports', 5: 'General', 6: 'Health'
}

dev_df['label_name'] = dev_df['label'].map(label_map)

fig = plt.figure(figsize=(10, 5))
sns.countplot(x='label_name', data=dev_df, 
              order=dev_df['label_name'].value_counts().index, 
              palette='viridis', hue='label_name', legend=False)

plt.title("Distribution of news categories")
plt.xticks(rotation=45)

fig.savefig("distribution_news_categories.pdf", bbox_inches="tight") 

plt.show()

print(dev_df['label'].value_counts(normalize=True))

Gestione testo e selezione feature utili. <br>

Gestione dei Duplicati (Solo nel Training)
Se nel development.csv  ci sono righe identiche, il modello va in overfitting su quelle frasi. 
Dobbiamo rimuovere i duplicati dal set di training, ma MAI dal set di evaluation (perché dobbiamo predire per ogni ID richiesto).

In [ ]:
# Duplicates
duplicates = dev_df.duplicated(subset=['title', 'article']).sum()
print(f"Duplicati trovati (Titolo+Articolo): {duplicates}")

inconsistent = dev_df.groupby(['title', 'article'])['label'].nunique()
print(inconsistent[inconsistent > 1])

# Controllo Lunghezza Testo
dev_df['text_len'] = (dev_df['title'].fillna('') + " " + dev_df['article'].fillna('')).apply(len)
print(f"Articoli con lunghezza 0 o < 50 chars: {(dev_df['text_len'] < 50).sum()}")

print("\n--- VERIFICA 'SOURCE' ---")
num_sources = dev_df['source'].nunique()
print(f"Numero totale di fonti uniche: {num_sources}")

print("\nTop 5 fonti più frequenti:")
print(dev_df['source'].value_counts().head(5))

print("\nQuante fonti appaiono meno di 5 volte?")
rare_sources = (dev_df['source'].value_counts() < 5).sum()
print(f"{rare_sources} su {num_sources} ({rare_sources/num_sources:.1%})")


def preprocess_text(df, remove_duplicates=False):
    if remove_duplicates:
        initial_len = len(df)
        
        # LOGICA AVANZATA DI GESTIONE DUPLICATI
        if 'label' in df.columns:
            # 1. Calcoliamo la frequenza di ogni label per ogni testo
            df['count_per_label'] = df.groupby(['title', 'article', 'label'])['title'].transform('count')
            
            # 2. Troviamo la frequenza MASSIMA per quel testo (indipendentemente dalla label)
            df['max_count_for_text'] = df.groupby(['title', 'article'])['count_per_label'].transform('max')
            
            # 3. Step Filtro: Teniamo solo le righe che hanno la label vincente (maggioranza)
            df_majority = df[df['count_per_label'] == df['max_count_for_text']].copy()
            
            # 4. Step Tie-Breaker: Gestione dei Pareggi (es. 1 Sport vs 1 Business)
            # Se dopo il filtro sopra, un testo ha ancora PIÙ di 1 label diversa, significa che c'era un pareggio.
            # Li identifichiamo e li rimuoviamo.
            inconsistent_mask = df_majority.duplicated(subset=['title', 'article'], keep=False)
            ties = df_majority[inconsistent_mask].groupby(['title', 'article'])['label'].nunique()
            ties_indices = ties[ties > 1].index
            
            # Rimuoviamo i testi che sono ancora ambigui (pareggi)
            df_clean = df_majority.set_index(['title', 'article']).drop(index=ties_indices, errors='ignore').reset_index()
            
            # 5. Pulizia Finale: Ora che abbiamo risolto i conflitti, teniamo 1 sola copia per testo
            df = df_clean.drop_duplicates(subset=['title', 'article'], keep='first').copy()
            
            # Rimuoviamo colonne temporanee
            drop_cols = ['count_per_label', 'max_count_for_text']
            df = df.drop(columns=[c for c in drop_cols if c in df.columns])
            
        else:
            df = df.drop_duplicates(subset=['title', 'article'], keep='first').copy()

        print(f"Rimossi {initial_len - len(df)} duplicati (Strategia Ibrida: Majority Vote + Tie Removal).")
        
    else:
        df = df.copy()
    
        
    # Feature weighting    
    source_feature = (df['source'].fillna('') + " ") * 3
    title_weighted  = (df['title'].fillna('') + " ") * 2
    df['text_combined'] = source_feature + title_weighted + " " + df['article'].fillna('')
    
    # Stemming
    stemmer = SnowballStemmer("english")
    
    def clean(text):
        text = str(text).lower()
        text = html.unescape(text)
        text = re.sub(r'<[^>]+>', ' ', text)
        text = re.sub(r'[^a-z0-9\s$€%£]', ' ', text) # useful symbols for class=business (€,%,$)                  
        text = re.sub(r'\s+', ' ', text).strip()
        return " ".join([stemmer.stem(word) for word in text.split()]) # stemming

    df['clean_text'] = df['text_combined'].apply(clean)
    return df


def extract_time_features(df):
    df = df.copy()
    
    # Convertiamo in datetime, gestendo gli errori
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    # 1. Feature Booleana: "Ha una data valida?" (Molto importante per Classe 2 vs 4)
    df['has_date'] = df['timestamp_dt'].notna().astype(int)
    
    # 2. Estraiamo ora e giorno solo dove possibile, altrimenti mettiamo un valore "neutro" (-1)
    # Usiamo -1 così il modello capisce che è diverso da "0" (mezzanotte)
    df['hour'] = df['timestamp_dt'].dt.hour.fillna(-1)
    df['day_of_week'] = df['timestamp_dt'].dt.dayofweek.fillna(-1)
    
    # Rimuoviamo la colonna temporanea dt e quella originale stringa se non serve più
    df = df.drop(columns=['timestamp_dt'])
    
    return df

dev_df['article'] = dev_df['article'].replace('\\N', '')
eval_df['article'] = eval_df['article'].replace('\\N', '')
print(f"Righe con \\N rimaste: {(dev_df['article'] == '\\N').sum()}")

dev_df = extract_time_features(dev_df)
eval_df = extract_time_features(eval_df) 

dev_df = preprocess_text(dev_df, remove_duplicates=True)
eval_df = preprocess_text(eval_df, remove_duplicates=False) 

dev_df = dev_df[dev_df['clean_text'].str.len() > 20].copy()
print(f"Rimossi articoli troppo corti. Nuova dimensione dev_df: {dev_df.shape}")